# 05 - Writing Small Outputs

IceTray `.i3` files are the native IceCube event format. They are powerful, but they are not always the most convenient format for quick plotting.

This notebook shows two common output patterns:

1. Use `I3HDFWriter` to write selected frame keys to HDF5.
2. Build a small Pandas table yourself and save it to HDF5.

Both approaches are useful. The second one is more transparent for beginners, so we write it out carefully.


In [ ]:
from pathlib import Path
import os

import pandas as pd
from icecube import dataio, dataclasses
from I3Tray import I3Tray
from icecube.hdfwriter import I3HDFWriter

DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)

print('Data file exists:', DATA_FILE.exists())
print('Output directory:', output_dir)


## Option A: I3HDFWriter

`I3HDFWriter` is an IceTray tool. It reads frames from a tray and writes selected frame keys to an HDF5 file.

This is convenient when the objects you want are already supported by the writer. Here we start with `I3EventHeader` and `QFilterMask`.


In [ ]:
hdf_output = output_dir / 'selected_keys_from_i3hdfwriter.hdf5'

tray = I3Tray()
tray.AddModule('I3Reader', 'reader', FilenameList=[str(DATA_GCD), str(DATA_FILE)])
tray.AddSegment(
    I3HDFWriter,
    'hdf_writer',
    Output=str(hdf_output),
    Keys=['I3EventHeader', 'QFilterMask'],
    SubEventStreams=['InIceSplit'],
)

print('Writing a small HDF5 file with I3HDFWriter...')
tray.Execute(500)
tray.Finish()
print('Wrote:', hdf_output)


## Option B: Make your own flat table

For early analysis work, it is often easier to create a table with one row per event. This table will not contain every IceCube object. It will contain only the simple values we choose.

The functions below repeat ideas from earlier notebooks on purpose: find Physics frames, find a pulse map, count DOMs, and add up charge.


In [ ]:
def frame_stop(frame):
    names = {'Q': 'DAQ', 'P': 'Physics', 'G': 'Geometry', 'C': 'Calibration', 'D': 'DetectorStatus', 'I': 'TrayInfo'}
    return names.get(str(frame.Stop), str(frame.Stop))

def find_pulse_key(frame):
    for key in ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']:
        if key in frame:
            return key
    return None

def pulse_summary(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        return {'pulse_key': None, 'hit_doms': None, 'number_of_pulses': None, 'total_charge': None}

    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)
    hit_doms = 0
    number_of_pulses = 0
    total_charge = 0.0

    for omkey, pulses_on_one_dom in pulse_map:
        hit_doms += 1
        number_of_pulses += len(pulses_on_one_dom)
        for pulse in pulses_on_one_dom:
            total_charge += pulse.charge

    return {'pulse_key': pulse_key, 'hit_doms': hit_doms, 'number_of_pulses': number_of_pulses, 'total_charge': total_charge}

print('Defined simple functions for building a flat table.')


## Loop through events and collect rows

Each row is a dictionary. The dictionary keys become table column names. This is a beginner-friendly pattern that works well for many first analyses.


In [ ]:
rows = []
physics_seen = 0
MAX_PHYSICS_FRAMES = 1000

i3_file = dataio.I3File(str(DATA_FILE))
while i3_file.more() and physics_seen < MAX_PHYSICS_FRAMES:
    frame = i3_file.pop_frame()
    if frame_stop(frame) != 'Physics':
        continue

    physics_seen += 1
    row = {}

    if 'I3EventHeader' in frame:
        header = frame['I3EventHeader']
        row['run_id'] = header.run_id
        row['event_id'] = header.event_id
        row['sub_event_id'] = header.sub_event_id
        row['sub_event_stream'] = str(header.sub_event_stream)

    row.update(pulse_summary(frame))
    rows.append(row)

i3_file.close()

event_table = pd.DataFrame(rows)
print(f'Built a table with {len(event_table)} rows.')
print('Columns:', list(event_table.columns))
event_table.head()


## Save and read back the table

Pandas can write a DataFrame to HDF5 with `to_hdf`. Reading it back is a good sanity check.


In [ ]:
custom_output = output_dir / 'event_hit_summary_from_pandas.h5'
event_table.to_hdf(custom_output, key='events', mode='w')

print('Saved table to:', custom_output)
print('Reading it back now:')
pd.read_hdf(custom_output, key='events').head()


## Practice

1. Add a column for reconstructed zenith if `SplineMPE` exists in the frame.
2. Add a column that says whether `MuonFilter_13` passed.
3. Change `MAX_PHYSICS_FRAMES` and compare file size.
4. Open the HDF5 file with Pandas in a new notebook and make a histogram from it.
